In [56]:
import numpy as np
from scipy.optimize import fsolve, root, least_squares, brentq

![dissipator circuit](full_diss_circuit.svg)

# Constants

In [57]:
e = 1.602176634e-19   # elementary charge, C
h = 6.626e-34          # Planck constant, J*s
hbar = 1.05457e034  # reduced Planck constant, J*s

fF = 1e-15
nH = 1e-9

# 1. Dissipator-Qubit Coupling

![dissipator circuit](qubit_dissipator_coupling.svg)

## A. Define Functions (Equations)

$$ E_C^Q = \frac{e^2 C_D+C_J}{2 C_D [ C_Q +C_J ]+C_QC_J}$$


$$E_C^D = \frac{e^2 C_Q +C_J}{2 C_D [ C_Q +C_J ]+C_QC_J}$$

In [58]:
def ECQ(CJ, CQ, CD):
    return (e**2 / 2) * ((CD + CJ) / (CD * CQ + CD * CJ + CQ * CJ))


def ECD(CJ, CQ, CD):
    return (e**2 / 2) * ((CQ + CJ) / (CD * CQ + CD * CJ + CQ * CJ))


and with a desired $\frac{E_J}{E_C}$ ratio, we can get $E_J$ by itself

In [59]:
def EJD(ratioD, CJ, CQ, CD):
    return ratioD * ECD(CJ, CQ, CD)


def EJQ(ratioQ, CJ, CQ, CD):
    return ratioQ * ECQ(CJ, CQ, CD)

with denoted transmon frequencies as: 
$$\omega_Q  \approx \sqrt{8E_J^QE_C^Q}-E_C^Q$$
$$\omega_D \approx \sqrt{8E_J^DE_C^D}-E_C^D$$

In [60]:
def omegaQ(ratioQ,CJ, CQ, CD):
    return np.sqrt(8 * ECQ(CJ, CQ, CD) * EJQ(ratioQ, CJ, CQ, CD)) - ECQ(CJ, CQ, CD)


def omegaD(ratioD,CJ, CQ, CD):
    return np.sqrt(8 * ECD(CJ, CQ, CD) * EJD(ratioD,CJ, CQ, CD)) - ECD(CJ, CQ, CD)

where anharmonicities are defined as:
$$\eta_Q \approx - E_C^Q$$
$$\eta_D \approx - E_C^D$$

coupling $J$ is then,
$$ J = -4e^2 \frac{C_J}{C_J C_Q + C_J C_D+C_D C_D}\left( \frac{E_J^Q}{32 E_C^Q} \frac{E_J^D}{32E_C^D}\right)^{1/4} $$

then we now need to be able to solve for a $C_J$ that gives us the desired $J$.

We typically have the $E_J/E_C$ ratios in mind beforehand, as well as the desired qubit/dissipator frequencies.

In [61]:
def J(ratioQ, ratioD, CJ, CQ, CD):
    return (
       ( -4
        * e**2)
        * (CJ / (CD * CQ + CD * CJ + CQ * CJ))
        * ( EJQ(ratioQ, CJ, CQ, CD) * EJD(ratioD, CJ, CQ, CD)/ (32*ECQ(CJ, CQ, CD)*32*ECD(CJ, CQ, CD))) ** (1 / 4)
    )


## B. Solver

### Target $E_J/E_C$ ratios

sometimes your $E_J/E_C$ ratio can be determined by your desired anharmonicity and qubit frequency

In [62]:
targetOmegaQ = 4.5e9 # Qubit frequency
targetOmegaD = 7.0e9 # Dissipator Frequency
targetEtaQ = -200e6 # anharmonicity


In [63]:
def solve_EJ_ratio(omega,eta):
    E_C = -eta
    E_J = (omega + E_C)**2 / 8/ E_C

    print(f'Target EJ: {E_J/1e9} GHz')
    print(f'Target EJ/EC: {E_J/E_C}')

    return E_J/E_C

In [64]:
ratioQ = solve_EJ_ratio(targetOmegaQ,targetEtaQ)

Target EJ: 13.80625 GHz
Target EJ/EC: 69.03125


and other times, we have a desired $E_J/E_C$ ratio at a specified frequency,

In [65]:
ratioD = 100 # from given parameters

In [66]:
def solve_EC_from_ratio(omega,ratio):
    E_C = omega / (np.sqrt(8*ratio)-1)

    print(f'Target EC: {E_C/1e6} MHz')
    return E_C

In [67]:
EC_D = solve_EC_from_ratio(targetOmegaD,ratioD)

Target EC: 256.55807100404667 MHz


Convert from frequencies to energy units

In [68]:
targetOmegaQh = targetOmegaQ * h # Qubit frequency
targetOmegaDh = targetOmegaD * h # Dissipator Frequency
targetJh =  -200e6 * h # Target coupliing

In [69]:
def equations(x_fF,targetOmegaQ,targetOmegaD,targetJ,ratioQ,ratioD):

    CJ, CQ, CD = [v * 1e-15 for v in x_fF]
    return [
        (omegaQ(ratioQ,CJ, CQ, CD) - targetOmegaQ) / targetOmegaQ,
        (omegaD(ratioD,CJ, CQ, CD) - targetOmegaD) / targetOmegaD,
        (J(ratioQ, ratioD, CJ, CQ, CD)- targetJ) / targetJ,
    ]

Provide initial guesses in femtoFarads

we can figure out relative guesses for capacitances given frequencies, and $E_J$ or $E_C$
using this site:

[Click here to visit Transmon Qubit Calculator](https://antonpotocnik.com/?p=560257)

In [70]:
# qubit-diss capacitance, qubit capcaitance, dissipator capacitance inital guesses
x0_fF = [1.89, 97, 75]

Now we can solve:

In [71]:
sol_fF = fsolve(equations, x0_fF,args=(targetOmegaQh,targetOmegaDh,targetJh,ratioQ,ratioD), full_output=False)
CJsol, CQsol, CDsol = [v * 1e-15 for v in sol_fF]
print(f"CJ = {CJsol / 1e-15} fF")
print(f"CQ = {CQsol / 1e-15} fF")
print(f"CD = {CDsol / 1e-15} fF")


CJ = 5.88463374569198 fF
CQ = 91.42404058610025 fF
CD = 69.97240240286669 fF


## C. Verify

In [72]:
print(f"Verify omegaQ = {omegaQ(ratioQ,CJsol, CQsol, CDsol) / h / 1e9} GHz (target: {targetOmegaQ/1e9})")
print(f"Verify omegaD = {omegaD(ratioD,CJsol, CQsol, CDsol) / h / 1e9} GHz (target: {targetOmegaD/1e9})")
print(f"Verify J      = {J(ratioQ,ratioD,CJsol, CQsol, CDsol) / h / 1e6} MHz (target: {targetJh/h})")
print(f"Verify EJQ/ECQ = {EJQ(ratioQ,CJsol, CQsol, CDsol) / ECQ(CJsol, CQsol, CDsol)} (target: {ratioQ})")  # NOTE: the original notebook's Print statement says "target: 50" here even though ratioQ = 70 -- likely a leftover/typo in the original, kept as-is for fidelity
print(f"Verify EJD/ECD = {EJD(ratioD,CJsol, CQsol, CDsol) / ECD(CJsol, CQsol, CDsol)} (target: {ratioD})")



Verify omegaQ = 4.500000000000001 GHz (target: 4.5)
Verify omegaD = 6.999999999999998 GHz (target: 7.0)
Verify J      = -199.99999999999866 MHz (target: -200000000.0)
Verify EJQ/ECQ = 69.03125 (target: 69.03125)
Verify EJD/ECD = 100.0 (target: 100)


# 2. Qubit-Resonator Coupling

Now let's consider the case of a readout resonator capacitively coupled to a transmon qubit.

We can model the resonator as LC oscillator, and the circuit diagram looks like this:

![qubit-resonator circuit](qubit_resonator_coupling.svg)

The Hamiltonian for this circuit is given by:
$$    \hat{H} = \omega_q \hat{a}^\dagger\hat{a} +\frac{\eta_Q}{2} \hat{a}^\dagger\hat{a}(\hat{a}^\dagger\hat{a}-1) +\omega_r \hat{c}^\dagger \hat{c} +g(\hat{a}-\hat{a}^\dagger)(\hat{c}-\hat{c}^\dagger).$$



## A. Define Functions (Equations)

Where the resonator frequency is:
$$\omega_r = \sqrt{8 E_J \tilde{E}_C^R}$$
with
$$E_L = \frac{\hbar^2}{4e^2L}=\frac{\varphi_0}{L}$$

In [73]:
def EL(L):
    return (hbar**2) / (4 * e**2 * L)

The transmon (qubit) frequency is, again,
$$\omega_Q = \sqrt{8E_J\tilde{E}_C^Q} -\tilde{E}_C^Q$$
with anharmonicity
$$\eta_Q \approx  -\tilde{E}_C^Q$$
and
$$ \tilde{E}_C^Q = \frac{e^2}{2}\frac{C_R +C_g}{\tilde{C}_Q[C_R+C_g]+C_RC_g}$$

we also can denote 
$$ E_C^R  = \frac{c^2}{2}\frac{\tilde{C}_Q +C_g}{C_R(\tilde{C}_R+C_g)+C_QC_g}$$

In [74]:
def Ctot (CR, Cg, CQ):
    return CR * CQ + CR * Cg + CQ * Cg

def ECQr(CR, Cg, CQ):
    return (e**2 / 2) * (CR + Cg) / (Ctot(CR, Cg, CQ))

def ECR(CR, Cg, CQ):
    return (e**2 / 2) * (CQ + Cg) / (Ctot(CR, Cg, CQ))


def EJQr(ratioQ, CR, Cg, CQ):
    return ratioQ * ECQr(CR, Cg, CQ)

def omegaQr(ratioQ, CR, Cg, CQ):
    ec = ECQr(CR, Cg, CQ)
    ej = ratioQ * ec
    val = 8 * ec * ej
    return np.sqrt(val) - ec if val > 0 else np.nan

def omegaR(L, CR, Cg, CQ):
    val = 8 * EL(L) * ECR(CR, Cg, CQ)
    return np.sqrt(val) if val > 0 else np.nan



Finally, the coupling $g$ is given by:
$$g = -4e^2 \frac{C_g}{C_R\tilde{C}_Q+C_RC_g + \tilde{C}_QC_g} \left( \frac{E_L^R}{32E_C^R}\frac{E_J^Q}{32\tilde{E}_C^Q}\right)^{1/4} \left(1+\frac{\eta_Q}{2\omega_Q}\right)$$

In [75]:
def g(ratioQ, L, CR, Cg, CQ):
    ec  = ECQr(CR, Cg, CQ)
    ecr = ECR(CR, Cg, CQ)
    ej  = ratioQ * ec
    el  = EL(L)
    inside = (el / (32*ecr)) * (ej / (32*ec))
    if inside <= 0:
        return np.nan
    prefactor  = (-4*e**2) * (Cg / Ctot(CR, Cg, CQ))
    correction = 1 + (-ec) / (2 * omegaQr(ratioQ, CR, Cg, CQ))
    return prefactor * (inside**0.25) * correction

## B. Solver

In [76]:
def get_ratio(omegaQ_Hz, ECQ_Hz):
    return ((omegaQ_Hz + ECQ_Hz)**2) / (8 * ECQ_Hz**2)

In [77]:
def make_initial_guess(omegaQ_Hz, omegaR_Hz, ECQ_Hz, g_Hz):


    CQ0 = (e**2 / (2 * ECQ_Hz * h)) / fF

    L_seed   = 2e-9
    EL_seed  = hbar**2 / (4 * e**2 * L_seed)
    ECR_seed = (omegaR_Hz * h)**2 / (8 * EL_seed)
    CR0      = (e**2 / (2 * ECR_seed)) / fF

    Cg0 = max(0.1, abs(g_Hz) / omegaQ_Hz * 10)

    ECR0 = (e**2/2) / (CR0 * fF)
    L0   = hbar**2 / (4 * e**2 * (omegaR_Hz * h)**2 / (8 * ECR0)) / nH

    CQ0 = np.clip(CQ0,  10,  600)
    CR0 = np.clip(CR0,  30, 3000)
    Cg0 = np.clip(Cg0, 0.05,  30)
    L0  = np.clip(L0,  0.05, 200)

    return CQ0, CR0, Cg0, L0

In [78]:
def find_bracket(func, lo, hi, n=500):
    """Scan [lo,hi] and return the first sub-interval with a sign change."""
    xs = np.linspace(lo, hi, n)
    ys = np.array([func(x) for x in xs])
    finite = np.isfinite(ys)
    if finite.sum() < 2:
        return False, None, None
    xs_f, ys_f = xs[finite], ys[finite]
    idx = np.where(np.diff(np.sign(ys_f)))[0]
    if len(idx) == 0:
        return False, None, None
    return True, xs_f[idx[0]], xs_f[idx[0]+1]


In [ ]:

def solve(targetOmegaQ_Hz, targetOmegaR_Hz, targetECQ_Hz, targetg_Hz,
          verbose=True):
    """
    Solve for (CQ, CR, Cg, L) given target frequencies and energies.

    Supported ranges
    ----------------
    fQ  : 3  – 10  GHz
    fR  : 4  – 11  GHz   (must be > fQ)
    ECQ : 100 – 300 MHz
    g   : any negative value in MHz range
    """

    # targets in Joules
    targetOmegaQ = targetOmegaQ_Hz * h
    targetOmegaR = targetOmegaR_Hz * h
    targetECQ    = targetECQ_Hz    * h
    targetg      = targetg_Hz      * h
    ratioQ       = get_ratio(targetOmegaQ_Hz, targetECQ_Hz)

    if verbose:
        print(f"── Targets ──────────────────────────────")
        print(f"  fQ  = {targetOmegaQ_Hz/1e9:.3f} GHz")
        print(f"  fR  = {targetOmegaR_Hz/1e9:.3f} GHz")
        print(f"  ECQ = {targetECQ_Hz/1e6:.1f} MHz")
        print(f"  g   = {targetg_Hz/1e6:.1f} MHz")
        print(f"  EJ/EC (derived) = {ratioQ:.4f}")

    # initialise from physics-based guess
    CQ0, CR0, Cg0, L0 = make_initial_guess(
        targetOmegaQ_Hz, targetOmegaR_Hz, targetECQ_Hz, targetg_Hz)

    CQ = CQ0 * fF
    CR = CR0 * fF
    Cg = Cg0 * fF
    L  = L0  * nH

    if verbose:
        print(f"\n── Initial guess ────────────────────────")
        print(f"  CQ = {CQ0:.2f} fF")
        print(f"  CR = {CR0:.2f} fF")
        print(f"  Cg = {Cg0:.3f} fF")
        print(f"  L  = {L0:.3f} nH")

    # ── Sequential iteration ──────────────────────────────────────────
    converged = False
    for iteration in range(500):

        # (a) CR → ECQ
        def res_ECQ(CR_fF):
            return ECQr(CR_fF*fF, Cg, CQ) - targetECQ
        ok, lo, hi = find_bracket(res_ECQ, 5, 15000)
        if ok:
            CR = brentq(res_ECQ, lo, hi, xtol=1e-8, rtol=1e-12) * fF

        # (b) CQ → omegaQ
        def res_omegaQ(CQ_fF):
            v = omegaQr(ratioQ, CR, Cg, CQ_fF*fF)
            return (v - targetOmegaQ) if np.isfinite(v) else 1e30
        ok, lo, hi = find_bracket(res_omegaQ, 5, 3000)
        if ok:
            CQ = brentq(res_omegaQ, lo, hi, xtol=1e-8, rtol=1e-12) * fF

        # (c) L → omegaR
        def res_omegaR(L_nH):
            v = omegaR(L_nH*nH, CR, Cg, CQ)
            return (v - targetOmegaR) if np.isfinite(v) else 1e30
        ok, lo, hi = find_bracket(res_omegaR, 0.001, 2000)
        if ok:
            L = brentq(res_omegaR, lo, hi, xtol=1e-10, rtol=1e-12) * nH

        # (d) Cg → g
        def res_g(Cg_fF):
            v = g(ratioQ, L, CR, Cg_fF*fF, CQ)
            return (v - targetg) if np.isfinite(v) else 1e30
        ok, lo, hi = find_bracket(res_g, 0.001, 500)
        if ok:
            Cg = brentq(res_g, lo, hi, xtol=1e-10, rtol=1e-12) * fF

        # convergence check
        gv = g(ratioQ, L, CR, Cg, CQ)
        if not np.isfinite(gv):
            continue

        r1 = abs(ECQr(CR,Cg,CQ)           - targetECQ)    / abs(targetECQ)
        r2 = abs(omegaQr(ratioQ,CR,Cg,CQ) - targetOmegaQ) / abs(targetOmegaQ)
        r3 = abs(omegaR(L,CR,Cg,CQ)      - targetOmegaR) / abs(targetOmegaR)
        r4 = abs(gv                       - targetg)       / abs(targetg)
        max_res = max(r1, r2, r3, r4)

        if verbose and iteration % 20 == 0:
            print(f"  iter {iteration:3d} | max_res={max_res:.2e} | "
                  f"ECQ={ECQr(CR,Cg,CQ)/h/1e6:.2f}MHz "
                  f"fQ={omegaQr(ratioQ,CR,Cg,CQ)/h/1e9:.4f}GHz "
                  f"fR={omegaR(L,CR,Cg,CQ)/h/1e9:.4f}GHz "
                  f"g={gv/h/1e6:.3f}MHz")

        if max_res < 1e-8:
            converged = True
            break

    # ── Polish with least_squares ─────────────────────────────────────
    x0 = [CQ/fF, CR/fF, Cg/fF, L/nH]
    # bounds are ±90% / 10x around sequential solution — always valid
    lower = [v * 0.1  for v in x0]
    upper = [v * 10.0 for v in x0]

    def residuals_polish(x_sc):
        CQp, CRp, Cgp, Lp = x_sc[0]*fF, x_sc[1]*fF, x_sc[2]*fF, x_sc[3]*nH
        gv = g(ratioQ, Lp, CRp, Cgp, CQp)
        if not np.isfinite(gv):
            return [1e6]*4
        return [
            (ECQr(CRp,Cgp,CQp)           - targetECQ)    / abs(targetECQ),
            (omegaQr(ratioQ,CRp,Cgp,CQp) - targetOmegaQ) / abs(targetOmegaQ),
            (omegaR(Lp,CRp,Cgp,CQp)     - targetOmegaR) / abs(targetOmegaR),
            (gv                          - targetg)       / abs(targetg),
        ]

    try:
        res = least_squares(residuals_polish, x0,
                            bounds=(lower, upper),
                            method='trf',
                            ftol=1e-14, xtol=1e-14, gtol=1e-14,
                            max_nfev=50000)
        CQ, CR, Cg, L = [v*u for v,u in zip(res.x, [fF,fF,fF,nH])]
    except Exception as ex:
        if verbose:
            print(f"  Polish skipped: {ex}")

    gv = g(ratioQ, L, CR, Cg, CQ)

    # ── Final residuals ───────────────────────────────────────────────
    r1 = abs(ECQr(CR,Cg,CQ)/h/1e6           - targetECQ_Hz/1e6)    / (targetECQ_Hz/1e6)
    r2 = abs(omegaQr(ratioQ,CR,Cg,CQ)/h/1e9 - targetOmegaQ_Hz/1e9) / (targetOmegaQ_Hz/1e9)
    r3 = abs(omegaR(L,CR,Cg,CQ)/h/1e9      - targetOmegaR_Hz/1e9) / (targetOmegaR_Hz/1e9)
    r4 = abs(gv/h/1e6 - targetg_Hz/1e6) / abs(targetg_Hz/1e6) if np.isfinite(gv) else np.nan
    max_res = max(r for r in [r1,r2,r3,r4] if np.isfinite(r))

    if verbose:
        print(f"\n── Solution ─────────────────────────────")
        print(f"  CQ = {CQ/fF:.4f} fF")
        print(f"  CR = {CR/fF:.4f} fF")
        print(f"  Cg = {Cg/fF:.4f} fF")
        print(f"  L  = {L/nH:.4f}  nH")
        print(f"\n── Verification ─────────────────────────")
        print(f"  ECQ = {ECQr(CR,Cg,CQ)/h/1e6:.4f} MHz   (target: {targetECQ_Hz/1e6:.1f})")
        print(f"  fQ  = {omegaQr(ratioQ,CR,Cg,CQ)/h/1e9:.4f} GHz   (target: {targetOmegaQ_Hz/1e9:.3f})")
        print(f"  fR  = {omegaR(L,CR,Cg,CQ)/h/1e9:.4f} GHz   (target: {targetOmegaR_Hz/1e9:.3f})")
        print(f"  g   = {gv/h/1e6:.4f} MHz   (target: {targetg_Hz/1e6:.1f})")
        print(f"\n  max_residual = {max_res:.2e}  {'✓ converged' if max_res < 1e-4 else '✗ check targets'}")

    return {
        "CQ_fF": CQ/fF, "CR_fF": CR/fF, "Cg_fF": Cg/fF, "L_nH": L/nH,
        "EJ_EC": ratioQ,
        "ECQ_MHz": ECQr(CR,Cg,CQ)/h/1e6,
        "fQ_GHz" : omegaQr(ratioQ,CR,Cg,CQ)/h/1e9,
        "fR_GHz" : omegaR(L,CR,Cg,CQ)/h/1e9,
        "g_MHz"  : gv/h/1e6 if np.isfinite(gv) else np.nan,
        "max_res": max_res,
        "converged": max_res < 1e-4,
    }


### Target Parameters (in energy units)

In [80]:
targetOmegaQ_Hz = 4.5e9
targetOmegaR_Hz = 5.75e9
targetECQ_Hz    = 200e6
targetg_Hz      = -50e6

targetOmegaQ = targetOmegaQ_Hz * h
targetOmegaR = targetOmegaR_Hz * h
targetECQ    = targetECQ_Hz    * h
targetg      = targetg_Hz      * h


ratioQ = solve_EJ_ratio(targetOmegaQ_Hz, targetECQ_Hz)

Target EJ: -11.55625 GHz
Target EJ/EC: 57.78125


We can use the same method that we used for the Dissipator-Qubit capacitance calculations to guess an initial CQ

### Solve

In [81]:
solve(5e9,7e9,200e6,50e6,False)

NameError: name 'g_func' is not defined